In [ ]:
import pandas as pd
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import snowballstemmer
from scipy import stats
from wordcloud import WordCloud
from collections import Counter
import random
import seaborn as sns
from sklearn.preprocessing import normalize
import matplotlib.patches as mpatches

# Duomenų įkėlimas ir paruošimas darbui

Įsikeliame failus. Nomenklatūrą sutvarkome į tris atskirus failus, kur iš hierachinio pavidalo padarome flat su reikalingais stulpeliais.


In [ ]:
def clean_goods_code(code):
    if pd.isna(code):
        return None
    code_str = str(code).replace(" ", "")
    return code_str[:10]

def get_indent_depth(indent):
    if pd.isna(indent) or str(indent).strip() == '':
        return 0
    return str(indent).count('-')

print("Sukurti failai: ")

def process_kn_file(file_path, output_path):
    df = pd.read_excel(file_path)
    df.columns = df.columns.str.strip()
    df["Goods code"] = df["Goods code"].apply(clean_goods_code)
    df["indent_depth"] = df["Indent"].apply(get_indent_depth)
    df = df.reset_index(drop=True)

    is_leaf = []
    for i in range(len(df)):
        if i == len(df) - 1:
            is_leaf.append(True)
        else:
            curr_depth = df.loc[i, "indent_depth"]
            next_depth = df.loc[i + 1, "indent_depth"]
            is_leaf.append(next_depth <= curr_depth)

    df["is_leaf"] = is_leaf

    results = []
    seen_codes = set()
    stack = []

    for i, row in df.iterrows():
        depth = row["indent_depth"]
        desc = str(row["Description"]).strip()
        code = row["Goods code"]

        while stack and stack[-1][0] >= depth:
            stack.pop()
        stack.append((depth, desc))

        if row["is_leaf"] and pd.notna(code) and code != '' and code not in seen_codes:
            full_description = " | ".join([d for _, d in stack])
            results.append({
                "Goods code": code,
                "Full description": full_description,
                "Start date": row.get("Start date"),
                "End date": row.get("End date"),
                "Language": row.get("Language")
            })
            seen_codes.add(code)

    result_df = pd.DataFrame(results)
    result_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{output_path} ({len(result_df)} eilutės)")

files = {
    "KN 2023.xlsx": "KN_2023_flat.csv",
    "KN 2024_pilnas.xlsx": "KN_2024_flat.csv",
    "KN 2025_pilnas.xlsx": "KN_2025_flat.csv"
}

for input_file, output_file in files.items():
    process_kn_file(input_file, output_file)

Žaliojo srauto duomenis sudedame į vieną .csv failą, pasiliekame reikalingus stulpelius. Tai pat ištriname 2026 m. duomenis, kadangi neturime 2026 m. nomenklatūros pakeitimų.


In [ ]:
def normalize_file(file_path):
    df = pd.read_excel(file_path)
    df.columns = df.columns.str.strip()

    col_map = {}

    for col in df.columns:
        if "018." in col or "020." in col:
            col_map[col] = "Data"
        elif "071." in col or "063." in col:
            col_map[col] = "Kodas"
        elif "085." in col or "072." in col:
            col_map[col] = "Aprašymas"

    df = df.rename(columns=col_map)

    df = df[["Data", "Kodas", "Aprašymas"]].copy()

    df["Kodas"] = (
        df["Kodas"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str[:10]
    )

    return df

files = [
    "Importo_prekes_lygio_2023_I_ketv_uzkod.xlsx",
    "Importo_prekes_lygio_2023_II_ketv uzkod.xlsx",
    "Importo_prekes_lygio_2023_III_ketv_uzkod.xlsx",
    "Importo_prekes_lygio_2023_IV_ketv_uzkod.xlsx",
    "Importo_prekes_lygio_20240311-20260315_uzkod.xlsx"
]

all_data = []

for file in files:
    df = normalize_file(file)
    all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True)

combined_df["Data"] = combined_df["Data"].astype(str)
combined_df = combined_df[~combined_df["Data"].str.startswith("2026")]

combined_df.to_csv("combined_imports_cleaned.csv", index=False, encoding="utf-8-sig")

print("Sukurtas failas:")
print("combined_imports_cleaned.csv")

Sutvarkom tekstą: viskas mažosiomis raidėmis, skaičiai (pvz., 0,5 mm ar 4/6 mm) išsaugomi kartu su matavimo vienetais, o skaičiai su kableliais visur pakeičiami į taškus. Pašalinami skyrybos ženklai, o lietuviškos raidės pakeičiamos paprastomis. Taip pat paliekami žodžių kamienai (t. y. nukertamos galūnės, kad „kriauklės“ ir „kriauklė“ būtų laikomi tuo pačiu žodžiu – „kriaukl“). Papildomai pašalinami dažni, mažai informatyvūs žodžiai (angl. stop words). Galiausiai iš deklaracijų duomenų pašalinamos besidubliuojančios eilutės, kai sutampa kodas ir aprašymas, bei pasižiūrime unikalių žodžių skaičių.

In [ ]:
kn_flat_input_files = ["KN_2023_flat.csv", "KN_2024_flat.csv", "KN_2025_flat.csv"]
combined_file = "combined_imports_cleaned.csv"

stemmer = snowballstemmer.stemmer('lithuanian')

lt_letter_map = str.maketrans({
    'ą': 'a', 'č': 'c', 'ę': 'e', 'ė': 'e',
    'į': 'i', 'š': 's', 'ų': 'u', 'ū': 'u', 'ž': 'z',
})

raw_stopwords = [
    'ir', 'iš', 'su', 'be', 'ar', 'bei', 'bet', 'tai', 'kad', 'jei',
    'per', 'prie', 'ant', 'už', 'iki', 'nuo', 'į', 'po', 'tarp', 'apie',
    'arba', 'nei', 'kaip', 'tik', 'dar', 'jau', 'ne', 'taip', 'kurie',
    'kurio', 'kurios', 'kurių', 'kuri', 'kuriam', 'kuriems', 'kuriuo',
    'kuriomis', 'kuriose', 'naudojamas', 'naudojami', 'naudojama', 'naudoti',
    'skirtas', 'skirti', 'skirta', 'skirtų', 'tipo', 'rūšies', 'rūšių'
]

normalized_stopwords = [w.lower().translate(lt_letter_map) for w in raw_stopwords]
stop_words = set(stemmer.stemWords(normalized_stopwords))

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = text.translate(lt_letter_map)
    text = re.sub(r'(\d+),(\d+)', r'\1.\2', text)
    text = re.sub(r'(\d+\.\d+)', lambda m: m.group(1).replace('.', 'XDOTX'), text)
    text = re.sub(r'[^\w\s\-_/]', ' ', text)
    text = text.replace('XDOTX', '.')
    text = re.sub(r'(\d+(?:[./]\d+)*)\s*(mm|cm|m|kg|g|l|ml|t|oz|in|ft)\b', r'\1_\2', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = stemmer.stemWords(tokens)
    tokens = [
        t for t in tokens
        if t not in stop_words
        and len(t) > 1
        and re.search(r'[a-z0-9]', t)
    ]
    return " ".join(tokens)

print("\nSukurti failai:")

for f in kn_flat_input_files:
    year = int(re.search(r'KN_(\d{4})_flat', f).group(1))
    df = pd.read_csv(f)
    df['cleaned_description'] = df['Full description'].apply(preprocess_text)

    output_file = f"KN_{year}_cleaned.csv"
    df.to_csv(output_file, index=False,
              columns=['Goods code', 'Full description', 'cleaned_description'])
    print(f"{output_file} ({len(df)} eilutės)")

combined_df = pd.read_csv(combined_file)
combined_df['cleaned_description'] = combined_df['Aprašymas'].apply(preprocess_text)

before = len(combined_df)

combined_df.to_csv(
    "combined_imports_cleaned_prepared_before_dedup.csv",
    index=False,
    columns=['Data', 'Kodas', 'Aprašymas', 'cleaned_description']
)
print(f"\ncombined_imports_cleaned_prepared_before_dedup.csv išsaugotas ({before} eilutės)")

combined_df = combined_df.drop_duplicates(subset=['Kodas', 'cleaned_description'])
after_dedup = len(combined_df)

combined_df.to_csv(
    "combined_imports_cleaned_prepared.csv",
    index=False,
    columns=['Data', 'Kodas', 'Aprašymas', 'cleaned_description']
)

print("\ncombined_imports_cleaned_prepared.csv:")
print(f"  Dublikatų pašalinta        : {before - after_dedup} ({before} → {after_dedup})")
print(f"  Galutinis eilučių skaičius : {after_dedup}")

print("\nUnikalių žodžių skaičius deklaracijos sutvarkytame ir pradiniame failuose:")

def get_words(series):
    return [w for text in series.dropna() for w in str(text).split()]

decl_words_clean = get_words(combined_df['cleaned_description'])
raw_decl_df = pd.read_csv("combined_imports_cleaned.csv")
decl_words_raw = get_words(raw_decl_df['Aprašymas'])

decl_clean_unique = len(set(decl_words_clean))
decl_raw_unique = len(set(decl_words_raw))
decl_clean_total = len(decl_words_clean)
decl_raw_total = len(decl_words_raw)

print(f"  pradinis — unikalių : {decl_raw_unique:>7} ({decl_raw_unique/decl_raw_total*100:.2f}%) | iš viso : {decl_raw_total:>9}")
print(f"  cleaned  — unikalių : {decl_clean_unique:>7} ({decl_clean_unique/decl_clean_total*100:.2f}%) | iš viso : {decl_clean_total:>9}")

Atliekame deklaracijų ir KN aprašymų tekstų analizę: lyginame žodžių vidurkius, standartinius nuokrypius, pasižiūrime KN kodų panaudojimą deklaracijose. Vizualizuojame dažniausiai pasitaikančius žodžius, sudarant žodžių debesis.

In [ ]:
kn_files = {
    2023: "KN_2023_cleaned.csv",
    2024: "KN_2024_cleaned.csv",
    2025: "KN_2025_cleaned.csv"
}
kn_flat_files = {
    2023: "KN_2023_flat.csv",
    2024: "KN_2024_flat.csv",
    2025: "KN_2025_flat.csv"
}
kn_raw_files = {
    2023: "KN 2023.xlsx",
    2024: "KN 2024_pilnas.xlsx",
    2025: "KN 2025_pilnas.xlsx"
}
combined_file = "combined_imports_cleaned_prepared.csv"

combined_df = pd.read_csv(combined_file)
kn_dfs = {year: pd.read_csv(path) for year, path in kn_files.items()}
kn_flat_dfs = {year: pd.read_csv(path) for year, path in kn_flat_files.items()}

combined_df['word_count'] = combined_df['cleaned_description'].str.split().str.len()
wc_groups = {'Deklaracijos': combined_df['word_count'].dropna()}

all_kn_wc = []

for year, df in kn_dfs.items():
    df['word_count'] = df['cleaned_description'].str.split().str.len()
    wc = df['word_count'].dropna()
    wc_groups[f'KN {year}'] = wc
    all_kn_wc.extend(wc)

all_kn_wc = pd.Series(all_kn_wc)

print("Žodžių skaičiaus statistika:")

for name, wc in wc_groups.items():
    print(f"{name:<15}  vidurkis={wc.mean():.2f}  std={wc.std():.2f}  n={len(wc)}")

decl_wc = wc_groups['Deklaracijos']

print("\nMann–Whitney U testas (Deklaracijos vs KN):")

for year in kn_dfs.keys():
    kn_wc = wc_groups[f'KN {year}']
    u, p = stats.mannwhitneyu(decl_wc, kn_wc, alternative='less')
    print(f"\nDeklaracijos vs KN {year}")
    print(f"U = {u:.4f}, p = {p:.2e}")

print("\n")

plot_df = pd.concat([
    pd.DataFrame({'word_count': decl_wc, 'grupė': 'Deklaracijos'}),
    pd.DataFrame({'word_count': all_kn_wc, 'grupė': 'Nomenklatūros'})
], ignore_index=True)

sns.set_theme(style="whitegrid", context="talk")

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

palette = {
    "Deklaracijos": "#50586C",
    "Nomenklatūros": "#50586C"
}

sns.boxplot(
    data=plot_df,
    x='grupė',
    y='word_count',
    hue='grupė',
    palette=palette,
    width=0.5,
    showfliers=True,
    dodge=False,
    legend=False,
    flierprops={
        "marker": "o",
        "markersize": 4,
        "markerfacecolor": "black",
        "markeredgecolor": "black",
        "alpha": 0.7
    },
    linewidth=1.5,
    medianprops={
        "color": "black",
        "linewidth": 2
    },
    whiskerprops={"linewidth": 1.2},
    capprops={"linewidth": 1.2},
    ax=ax
)

ax.set_title(
    "Žodžių skaičiaus pasiskirstymas deklaracijų ir nomenklatūrų failuose",
    fontsize=12,
    fontweight='bold',
    pad=12
)

ax.set_xlabel("")
ax.set_ylabel("Žodžių skaičius", fontsize=12, fontweight='bold')

ax.tick_params(axis='x', labelsize=12, colors='black')
ax.tick_params(axis='y', labelsize=11, colors='black')

for label in ax.get_xticklabels():
    label.set_fontweight('bold')
    label.set_color('black')

ax.grid(True, axis='y', linestyle='--', alpha=0.3, color='gray')
ax.grid(False, axis='x')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_color("gray")
ax.spines['bottom'].set_color("gray")

ax.spines['left'].set_linewidth(1.2)
ax.spines['bottom'].set_linewidth(1.2)

plt.tight_layout()
plt.show()

print("\n")

print("Panaudoti KN kodai deklaracijose (pagal metus):")
combined_df['year'] = pd.to_datetime(combined_df['Data'], errors='coerce').dt.year

for year, df in kn_dfs.items():
    kn_codes = set(df['Goods code'].dropna().astype(str).str.replace(" ", ""))
    decl_year = combined_df[combined_df['year'] == year]
    used_codes = set(decl_year['Kodas'].dropna().astype(str)) & kn_codes
    print(f"{year}: {len(used_codes)} panaudotų / {len(kn_codes)} KN kodų "
          f"({100 * len(used_codes) / len(kn_codes):.1f}%)")

print("\n")

print("Dažniausiai naudojamos KN klasės deklaracijose (pagal metus):\n")

for year, df in kn_flat_dfs.items():
    kn_codes = df[['Goods code', 'Full description']].dropna().copy()
    kn_codes['code_clean'] = kn_codes['Goods code'].astype(str).str.replace(" ", "")

    decl_year = combined_df[combined_df['year'] == year].copy()
    decl_year['code_clean'] = decl_year['Kodas'].dropna().astype(str)

    merged = decl_year.merge(kn_codes, left_on='code_clean', right_on='code_clean', how='inner')

    top5 = (
        merged.groupby(['code_clean', 'Full description'])
        .size()
        .sort_values(ascending=False)
        .head(5)
        .reset_index()
    )
    top5.columns = ['Kodas', 'Aprašymas', 'Kiekis']

    print(f"--- {year} ---")
    for i, row in top5.iterrows():
        print(f"  {i+1}. [{row['Kodas']}] {row['Aprašymas']}  ({row['Kiekis']})")
    print()

print("\n")

def shorten_label(label, all_labels, prefix_max=30):
    if label == 'Kita':
        return label

    parts = [p.strip() for p in label.split('|')]

    same_group = [
        l for l in all_labels
        if l != label and l != 'Kita'
        and l.split('|')[0].strip() == parts[0]
    ]

    if not same_group:
        if len(parts) == 1:
            return parts[0]
        return parts[0] + ' | ' + ' | '.join(parts[1:])

    all_parts = [[p.strip() for p in l.split('|')] for l in same_group]

    common_depth = 0
    for depth in range(len(parts)):
        if all(depth < len(op) and op[depth] == parts[depth] for op in all_parts):
            common_depth = depth + 1
        else:
            break

    if common_depth == 0 or common_depth >= len(parts):
        return parts[0]

    first_words = ' '.join(parts[0].split()[:3])
    unique_parts = ' | '.join(parts[common_depth:])
    return f"{first_words} <...> | {unique_parts}"


top_n = 5
all_counts = {}
all_labels_ordered = []

for i, (year, df) in enumerate(kn_flat_dfs.items()):
    kn_codes = df[['Goods code', 'Full description']].dropna().copy()
    kn_codes['code_clean'] = kn_codes['Goods code'].astype(str).str.replace(" ", "")

    decl_year = combined_df[combined_df['year'] == year].copy()
    decl_year['code_clean'] = decl_year['Kodas'].dropna().astype(str)

    merged = decl_year.merge(kn_codes, on='code_clean', how='inner')

    counts = (
        merged.groupby(['code_clean', 'Full description'])
        .size()
        .sort_values(ascending=False)
    )
    counts.index = [desc for _, desc in counts.index]

    top = counts.head(top_n)
    other_val = counts.iloc[top_n:].sum()
    counts_plot = pd.concat([top, pd.Series({'Kita': other_val})])
    all_counts[year] = counts_plot

    if i == 0:
        all_labels_ordered = list(counts_plot.index)

palette_colors = ["#1c1c1c", "#3a3a3a", "#5a5a5a", "#7a7a7a", "#a0a0a0", "#c8c8c8"]
color_map = {label: palette_colors[i] for i, label in enumerate(all_labels_ordered)}

non_kita = [l for l in all_labels_ordered if l != 'Kita']
legend_labels_display = [shorten_label(l, non_kita) for l in all_labels_ordered]

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

palette = {
    "Deklaracijos": "#50586C",
    "Nomenklatūros": "#50586C"
}

sns.boxplot(
    data=plot_df,
    x='grupė',
    y='word_count',
    hue='grupė',
    palette=palette,
    width=0.5,
    showfliers=True,
    dodge=False,
    legend=False,
    flierprops={
        "marker": "o",
        "markersize": 4,
        "markerfacecolor": "black",
        "markeredgecolor": "black",
        "alpha": 0.7
    },
    linewidth=1.5,
    medianprops={
        "color": "black",
        "linewidth": 2
    },
    whiskerprops={"linewidth": 1.2},
    capprops={"linewidth": 1.2},
    ax=ax
)

ax.set_title(
    "Žodžių skaičiaus pasiskirstymas deklaracijų ir nomenklatūrų failuose",
    fontsize=12,
    fontweight='bold',
    pad=12
)

ax.set_xlabel("")
ax.set_ylabel("Žodžių skaičius", fontsize=12, fontweight='bold')

ax.tick_params(axis='x', labelsize=12, colors='black')
ax.tick_params(axis='y', labelsize=11, colors='black')

for label in ax.get_xticklabels():
    label.set_fontweight('bold')
    label.set_color('black')

ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.grid(False, axis='x')

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('#cccccc')
    spine.set_linewidth(1.2)

plt.tight_layout()
plt.show()

print("\n")

def get_raw_words(series):
    return [w for text in series.dropna() for w in str(text).split() if len(w) > 1]

def dark_grey_color(*args, **kwargs):
    val = random.randint(0, 100)
    return f"rgb({val}, {val}, {val})"

def make_wordcloud(words, title):
    freq = Counter(words)

    wc = WordCloud(
        width=900,
        height=450,
        background_color='white',
        max_words=100,
        color_func=dark_grey_color
    ).generate_from_frequencies(freq)

    plt.figure(figsize=(12, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

decl_words_wc = get_raw_words(combined_df['cleaned_description'])

all_kn_words_wc = []
for year, df in kn_dfs.items():
    all_kn_words_wc.extend(get_raw_words(df['cleaned_description']))

make_wordcloud(decl_words_wc, "Deklaracijų žodžių debesis")
print("\n")
make_wordcloud(all_kn_words_wc, "Nomenklatūrų žodžių debesis (KN 2023–2025)")

## Uždaviniai

NLP naudojimas. Kode pirmiausia paruošiami deklaracijų ir KN nomenklatūros tekstiniai duomenys, iš kurių sukuriami TF-IDF vektoriai (žinoma, kad TF-IDF reikšmingai skiriasi, kai nesutampa tekstų ilgiai, nors trumpesnio teksto (deklaracijoje randamo prekės aprašymo) žodžiai ir atitinka kai kuriuos ilgesnio teksto (nomenklatūroje randamo prekės aprašymo) žodžius), leidžiantys tekstus paversti skaitine forma. Toliau kiekvienai deklaracijai ieškomi panašiausi nomenklatūros įrašai naudojant kelis skirtingus metodus: containment, kosinusinį panašumą, euklidinį atstumą. Containment metodas naudojamas įvertinti, kiek deklaracijos aprašymo žodžių sutampa su nomenklatūros aprašymu. Kosinusinis panašumas lygina TF-IDF vektorių kryptis ir padeda nustatyti tekstų semantinį panašumą. Euklidinis atstumas matuoja atstumą tarp TF-IDF vektorių, todėl mažesnė reikšmė reiškia didesnį tekstų panašumą. Kiekvienam metodui apskaičiuojami Top-1, Top-3 ir Top-5 tikslumo rodikliai, kurie parodo, ar teisingas prekės kodas buvo rastas tarp labiausiai panašių rezultatų.

In [ ]:
kn_files = {2023: "KN_2023_cleaned.csv", 2024: "KN_2024_cleaned.csv", 2025: "KN_2025_cleaned.csv"}
combined_file = "combined_imports_cleaned_prepared.csv"

combined_df = pd.read_csv(combined_file)
kn_dfs = {year: pd.read_csv(path) for year, path in kn_files.items()}
kn_dfs = {year: df.reset_index(drop=True) for year, df in kn_dfs.items()}

all_texts = pd.concat(
    [df['cleaned_description'] for df in kn_dfs.values()] +
    [combined_df['cleaned_description']],
    ignore_index=True
)
vectorizer = TfidfVectorizer(
    lowercase=False,
    token_pattern=r'\b[\w\-/\.]+\b',
    sublinear_tf=True
)
vectorizer.fit(all_texts)

kn_tfidf = {
    year: normalize(vectorizer.transform(df['cleaned_description']), norm='l2')
    for year, df in kn_dfs.items()
}

TOP_K = 5

def containment_score(decl_words, kn_text):
    if not decl_words or not kn_text:
        return 0.0
    return len(decl_words.intersection(set(kn_text.split()))) / len(decl_words)


def get_top_k(scores, kn_df, k, ascending=False):
    idx = np.argsort(scores)
    if not ascending:
        idx = idx[::-1]
    top_idx = idx[:k]
    return [(str(kn_df.loc[i, 'Goods code']), scores[i]) for i in top_idx]


def suggest_code(decl_row, kn_dfs, kn_tfidf, vectorizer,
                 methods=('containment', 'cosine', 'euclidean'), k=TOP_K):
    year = pd.to_datetime(decl_row['Data']).year
    if year not in kn_dfs:
        print(f"No KN data for year {year}")
        return

    decl_text = decl_row['cleaned_description']
    decl_text_raw = decl_row['Aprašymas']
    decl_code = str(decl_row['Kodas'])
    kn_df = kn_dfs[year]
    kn_matrix = kn_tfidf[year]

    results = {}

    if 'containment' in methods:
        decl_words = set(decl_text.split()) if decl_text else set()
        cont_scores = kn_df['cleaned_description'].apply(
            lambda t: containment_score(decl_words, t)
        ).values
        results['containment'] = get_top_k(cont_scores, kn_df, k, ascending=False)

    decl_vec = normalize(vectorizer.transform([decl_text]), norm='l2')

    if 'cosine' in methods:
        cos_scores = cosine_similarity(decl_vec, kn_matrix)[0]
        results['cosine'] = get_top_k(cos_scores, kn_df, k, ascending=False)

    if 'euclidean' in methods:
        euc_dists = euclidean_distances(decl_vec, kn_matrix)[0]
        results['euclidean'] = get_top_k(euc_dists, kn_df, k, ascending=True)

    print("=" * 80)
    print(f"Deklaracijos data       : {decl_row['Data']}")
    print(f"Deklaracijos kodas      : {decl_code}")
    print(f"Deklaracijos apraš.     : {decl_text_raw}")
    print(f"Deklaracijos su kamien. : {decl_text}")
    print()

    for method, top_list in results.items():
        label = "distance" if method == 'euclidean' else "score"
        top1_code = top_list[0][0]
        top1_match = "✓ SUTAMPA" if top1_code == decl_code else "✗ NESUTAMPA"
        in_top3 = any(code == decl_code for code, _ in top_list[:3])
        in_top5 = any(code == decl_code for code, _ in top_list)
        top3_flag = "✓ top3" if in_top3 else "✗ top3"
        top5_flag = "✓ top5" if in_top5 else "✗ top5"

        print(f"  [{method.upper()}]  top1: {top1_match}  |  {top3_flag}  |  {top5_flag}")
        for rank, (code, score) in enumerate(top_list, 1):
            match_flag = "✓" if code == decl_code else " "
            kn_row = kn_dfs[year][kn_dfs[year]['Goods code'].astype(str) == code]
            kn_stem = kn_row['cleaned_description'].values[0] if len(kn_row) else ""
            print(f"    #{rank} {match_flag} {code}  {label}={score:.6f}  stem: {kn_stem}")
        print()

N = 10
sample = combined_df.sample(n=N, random_state=42)

for _, row in tqdm(sample.iterrows(), total=N, desc="Suggesting codes"):
    suggest_code(row, kn_dfs, kn_tfidf, vectorizer,
                 methods=('containment', 'cosine', 'euclidean'), k=TOP_K)

eval_df = combined_df.reset_index(drop=True)

results_all = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
    year = pd.to_datetime(row['Data']).year
    if year not in kn_dfs:
        continue

    decl_text = row['cleaned_description']
    decl_code = str(row['Kodas'])
    kn_df = kn_dfs[year]
    kn_matrix = kn_tfidf[year]

    row_result = {'Kodas': decl_code, 'Data': row['Data']}

    decl_words = set(decl_text.split()) if decl_text else set()
    cont_scores = kn_df['cleaned_description'].apply(
        lambda t: containment_score(decl_words, t)
    ).values

    decl_vec = normalize(vectorizer.transform([decl_text]), norm='l2')
    cos_scores = cosine_similarity(decl_vec, kn_matrix)[0]
    euc_dists = euclidean_distances(decl_vec, kn_matrix)[0]

    for method, scores, asc in [
        ('containment', cont_scores, False),
        ('cosine',      cos_scores,  False),
        ('euclidean',   euc_dists,   True),
    ]:
        top = get_top_k(scores, kn_df, TOP_K, ascending=asc)
        row_result[f'{method}_top1'] = top[0][0] == decl_code
        row_result[f'{method}_top3'] = any(c == decl_code for c, _ in top[:3])
        row_result[f'{method}_top5'] = any(c == decl_code for c, _ in top)

    results_all.append(row_result)

results_df = pd.DataFrame(results_all)

n = len(results_df)
print("\n" + "=" * 50)
print(f"Iš viso įvertinta: {n} eilučių")
print("=" * 50)

for method in ('containment', 'cosine', 'euclidean'):
    top1 = results_df[f'{method}_top1'].sum()
    top3 = results_df[f'{method}_top3'].sum()
    top5 = results_df[f'{method}_top5'].sum()
    print(f"\n[{method.upper()}]")
    print(f"  ✓ Top-1 sutampa: {top1:>6} ({top1/n*100:.2f}%)")
    print(f"  ✓ Top-3 sutampa: {top3:>6} ({top3/n*100:.2f}%)")
    print(f"  ✓ Top-5 sutampa: {top5:>6} ({top5/n*100:.2f}%)")
    print(f"  ✗ Ne top-5     : {n-top5:>6} ({(n-top5)/n*100:.2f}%)")

print("=" * 50)

Aibių siaurinimas. Šiuose modeliuose pirmiausia apskaičiuojamas kiekvieno žodžio dažnis KN nomenklatūros aprašymuose, o tada deklaracijos žodžiai surikiuojami pagal jų retumą arba dažnumą. Toliau deklaracijos aprašymas naudojamas kandidatų filtravimui – paliekami tik tie nomenklatūros įrašai, kuriuose randami pasirinkti žodžiai. Iš gautų kandidatų kodų nustatoma ilgiausia bendra kodo pradžia (angl. common prefix), kuri parodo, kiek pirmųjų skaitmenų sutampa tarp visų rastų kandidatų kodų. Tuomet tikrinama, ar ši bendra kodo pradžia sutampa su tikruoju deklaracijos kodu. Rezultatai grupuojami pagal sutampančių skaitmenų lygį (2, 4, 6, 8 arba 10), o galutinis modelio tikslumas apskaičiuojamas naudojant svorius, didesnę reikšmę suteikiant ilgesniems ir tikslesniems kodo sutapimams. Taip pat palyginami du modeliai – rečiausių ir dažniausių žodžių naudojimas kandidatų filtravimui.

In [ ]:
N_EXAMPLES = 5
WEIGHTS = {4: 2, 6: 3, 8: 4, 10: 5}

kn_doc_freq = Counter()
for df in kn_dfs.values():
    for text in df['cleaned_description'].fillna(''):
        for word in set(str(text).split()):
            kn_doc_freq[word] += 1


def select_words(words, mode='rare'):
    words = list(set(words))
    if not words:
        return []
    reverse = (mode == 'common')
    return sorted(words, key=lambda w: kn_doc_freq.get(w, 0), reverse=reverse)


def filter_candidates_unlimited(kn_df, selected_words):
    candidates = kn_df.copy()
    for word in selected_words:
        new_candidates = candidates[
            candidates['cleaned_description']
            .str.contains(rf'\b{re.escape(word)}\b', regex=True, na=False)
        ]
        if len(new_candidates) >= 1:
            candidates = new_candidates
    return candidates


def common_prefix_len(codes):
    if len(codes) == 0:
        return 0
    if len(codes) == 1:
        return len(codes[0])
    ref = codes[0]
    length = 0
    for i in range(len(ref)):
        if all(c[i] == ref[i] for c in codes if i < len(c)):
            length += 1
        else:
            break
    return length


def get_eval_df():
    df = combined_df.copy()
    df['Year'] = pd.to_datetime(df['Data']).dt.year
    return df


def weighted_accuracy(buckets):
    weighted_correct = 0
    weighted_total   = 0
    for d, w in WEIGHTS.items():
        c   = buckets[d]['correct']
        tot = buckets[d]['correct'] + buckets[d]['wrong']
        if tot == 0:
            continue
        weighted_correct += c * w
        weighted_total   += tot * w
    return weighted_correct / weighted_total * 100 if weighted_total > 0 else 0.0


def print_examples(examples, model_label):
    print(f"\n  --- Veikimo pavyzdžiai ---")
    for i, ex in enumerate(examples, 1):
        print(f"\n  {i} pavyzdys:")
        print(f"    Deklaracijos aprašymas           : {ex['description']}")
        print(f"    Tikrasis kodas                   : {ex['true_code']}")
        print(f"    Naudoti žodžiai                  : {ex['selected_words']}")
        print(f"    Kandidatų skaičius               : {ex['n_cands']}")
        if 'common_prefix' in ex:
            print(f"    Bendra kodo pradžia              : {ex['common_prefix']!r} ({ex['cp_len']} skaitm.)")
            print(f"    Ar bendra kodo pradžia teisinga? : {'✓' if ex['cp_correct'] else '✗'}")
            print(f"    Kandidatų kodai (pvz.)           : {ex['sample_codes']}")


def evaluate_common_prefix(mode):
    df    = get_eval_df()
    total = len(df)

    buckets = {d: {'correct': 0, 'wrong': 0} for d in [2, 4, 6, 8, 10]}
    collected_examples = []


    for _, row in tqdm(df.iterrows(), total=total):
        year = row['Year']
        if year not in kn_dfs:
            continue

        kn_df   = kn_dfs[year]
        words   = str(row['cleaned_description']).split()
        true_10 = str(row['Kodas'])

        selected = select_words(words, mode=mode)
        cands    = filter_candidates_unlimited(kn_df, selected)

        if len(cands) == 0:
            buckets[2]['wrong'] += 1
            continue

        codes  = cands['Goods code'].astype(str).tolist()
        cp_len = common_prefix_len(codes)

        for lvl in [10, 8, 6, 4, 2]:
            if cp_len >= lvl:
                group = lvl
                break
        else:
            group = 2

        common_str = codes[0][:group] if group > 0 else ''
        correct    = true_10.startswith(common_str) if common_str else False

        if correct:
            buckets[group]['correct'] += 1
        else:
            buckets[group]['wrong']   += 1

        if len(collected_examples) < N_EXAMPLES * 10:
            collected_examples.append({
                'description'   : str(row['cleaned_description'])[:120],
                'true_code'     : true_10,
                'selected_words': selected[:8],
                'n_cands'       : len(cands),
                'common_prefix' : common_str,
                'cp_len'        : group,
                'cp_correct'    : correct,
                'sample_codes'  : codes[:6],
            })

    n = total or 1

    print(f"\n{'='*65}")
    print(f"  [{mode.upper()} | COMMON-PREFIX MODELIS] | n={total}")
    print(f"{'='*65}")
    print(f"\n  Bendra kodo pradžia (visų kodų X skaitmenys sutampa) + ar teisingas?:")
    print(f"    {'Lygis':>10}  {'Svoris':>6}  {'Teisingi':>10}  {'Neteisingi':>10}  {'Iš viso':>8}  {'%':>7}  {'Tikslumas':>10}")

    for d in [2, 4, 6, 8, 10]:
        c   = buckets[d]['correct']
        w   = buckets[d]['wrong']
        tot = c + w
        if tot == 0:
            continue
        pct   = tot / n * 100
        c_pct = c / tot * 100 if tot > 0 else 0.0
        print(f"    {d:>2} skaitm.  {WEIGHTS.get(d, '-'):>6}  {c:>10}  {w:>10}  {tot:>8}  ({pct:>5.2f}%)  [{c_pct:>5.1f}%]")

    wa = weighted_accuracy(buckets)
    print(f"\n  Tikslumas su svoriais: {wa:.2f}%")

    sample = random.sample(collected_examples, min(N_EXAMPLES, len(collected_examples)))
    print_examples(sample, f"{mode.upper()} COMMON-PREFIX")

    return buckets, total

buckets_rare,   total = evaluate_common_prefix(mode='rare')
buckets_common, _     = evaluate_common_prefix(mode='common')

wa_rare   = weighted_accuracy(buckets_rare)
wa_common = weighted_accuracy(buckets_common)

print(f"\n{'='*55}")
print(f"  MODELIŲ PALYGINIMAS")
print(f"{'='*55}")
print(f"  RARE   svertinis tikslumas: {wa_rare:>6.2f}%")
print(f"  COMMON svertinis tikslumas: {wa_common:>6.2f}%")
print(f"\n  Geresnis modelis: {'RARE' if wa_rare > wa_common else 'COMMON'} "
      f"(skirtumas: {abs(wa_rare - wa_common):.2f}%)")
print(f"{'-'*55}")
print(f"  Detalus palyginimas pagal lygius:")
print(f"    {'Lygis':>8}  {'RARE tiksl.':>16}  {'COMMON tiksl.':>18}")
for d, w in WEIGHTS.items():
    def pct(buckets, d=d):
        c   = buckets[d]['correct']
        tot = c + buckets[d]['wrong']
        return c / tot * 100 if tot > 0 else 0.0
    r = pct(buckets_rare)
    c = pct(buckets_common)
    winner = '← RARE' if r > c else ('← COMMON' if c > r else '=')
    print(f"    {d:>2} sk.      {r:>11.1f}%  {c:>13.1f}%  {winner}")

In [ ]:
def plot_split_bars(buckets_rare, buckets_common, total):
    levels = [2, 4, 6, 8, 10]
    labels = [f"{d} sk." for d in levels]

    n = total or 1

    data = {}
    for mode, buckets in [('RARE', buckets_rare), ('COMMON', buckets_common)]:
        fracs, accs = [], []
        for d in levels:
            c   = buckets[d]['correct']
            tot = c + buckets[d]['wrong']
            fracs.append(tot / n * 100)
            accs.append(c / tot * 100 if tot > 0 else 0.0)
        data[mode] = {'frac': fracs, 'acc': accs}

    x     = np.arange(len(levels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(12, 7))

    colors = {
        'RARE':   {'correct': '#1c1c1c', 'wrong': '#505050'},
        'COMMON': {'correct': '#8c8c8c', 'wrong': '#cccccc'},
    }

    for mode, offset in [('RARE', -width/2), ('COMMON', width/2)]:
        fracs = data[mode]['frac']
        accs  = data[mode]['acc']

        correct_heights = [f * a / 100 for f, a in zip(fracs, accs)]
        wrong_heights   = [f - ch for f, ch in zip(fracs, correct_heights)]

        ax.bar(x + offset, correct_heights, width,
               color=colors[mode]['correct'], zorder=3)
        ax.bar(x + offset, wrong_heights, width,
               bottom=correct_heights,
               color=colors[mode]['wrong'], zorder=3)

        for xi, (frac, acc, level) in enumerate(zip(fracs, accs, levels)):
            if frac > 0:
                correct_h = frac * acc / 100
                label = f'{acc:.1f}%' if acc < 100 else '100%'
                if level == 10:
                    y          = correct_h + 0.4
                    va         = 'bottom'
                    text_color = '#1c1c1c' if mode == 'COMMON' else 'white'
                else:
                    y          = correct_h * 0.5
                    va         = 'center'
                    text_color = 'white'
                ax.text(
                    xi + offset, y, label,
                    ha='center', va=va, fontsize=10,
                    color=text_color, fontweight='bold'
                )

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel('Bendros kodo pradžios ilgis', fontsize=11, fontweight='bold')
    ax.set_ylabel('Dalis nuo visų įrašų (%)', fontsize=11, fontweight='bold')
    ax.set_title(
        'Rare ir Common modelių palyginimas pagal prekės kodo pradžios nustatymo tikslumą',
        fontsize=14, fontweight='bold'
    )
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
    ax.set_axisbelow(True)

    handles = [
        mpatches.Patch(color=colors['RARE']['correct'],   label='$\it{Rare}$ – teisingi'),
        mpatches.Patch(color=colors['RARE']['wrong'],     label='$\it{Rare}$ – neteisingi'),
        mpatches.Patch(color=colors['COMMON']['correct'], label='$\it{Common}$ – teisingi'),
        mpatches.Patch(color=colors['COMMON']['wrong'],   label='$\it{Common}$ – neteisingi'),
    ]
    ax.legend(handles=handles, loc='upper right', fontsize=9)

    plt.tight_layout()
    plt.show()

plot_split_bars(buckets_rare, buckets_common, total)

Ši analizė vertina, kaip modelio tikslumas priklauso nuo deklaracijos aprašymo ilgio. Deklaracijos suskirstomos į grupes pagal žodžių skaičių, o kiekvienai grupei atskirai taikomas common-prefix modelis ((„rare“ arba „common“)). Kiekvienai žodžių ilgio grupei apskaičiuojamas bendras ir svertinis tikslumas, o rezultatai leidžia palyginti, kaip aprašymo ilgis bei skirtingi žodžių atrankos būdai veikia klasifikavimo kokybę.

In [ ]:
WORD_BINS  = [(1, 2), (3, 4), (5, 6), (7, 9), (10, 14), (15, 999)]
BIN_LABELS = {(1,2): "1–2", (3,4): "3–4", (5,6): "5–6", (7,9): "7–9", (10,14): "10–14", (15,999): "15+"}

def get_bin(n):
    for lo, hi in WORD_BINS:
        if lo <= n <= hi:
            return (lo, hi)
    return (15, 999)


def evaluate_common_prefix_by_wordcount(mode):
    df    = get_eval_df()
    total = len(df)

    results = {
        b: {d: {'correct': 0, 'wrong': 0} for d in [2, 4, 6, 8, 10]}
        for b in WORD_BINS
    }
    bin_counts = Counter()

    for _, row in tqdm(df.iterrows(), total=total):
        year = row['Year']
        if year not in kn_dfs:
            continue

        kn_df   = kn_dfs[year]
        words   = str(row['cleaned_description']).split()
        true_10 = str(row['Kodas'])
        n_words = len(words)
        b       = get_bin(n_words)
        bin_counts[b] += 1

        selected = select_words(words, mode=mode)
        cands    = filter_candidates_unlimited(kn_df, selected)

        if len(cands) == 0:
            results[b][2]['wrong'] += 1
            continue

        codes  = cands['Goods code'].astype(str).tolist()
        cp_len = common_prefix_len(codes)

        for lvl in [10, 8, 6, 4, 2]:
            if cp_len >= lvl:
                group = lvl
                break
        else:
            group = 2

        common_str = codes[0][:group] if group > 0 else ''
        correct    = true_10.startswith(common_str) if common_str else False

        if correct:
            results[b][group]['correct'] += 1
        else:
            results[b][group]['wrong']   += 1

    print(f"\n{'='*80}")
    print(f"  [{mode.upper()} | COMMON-PREFIX | pagal žodžių skaičių | n={total}]")
    print(f"  Svoriai: {dict(WEIGHTS)}")
    print(f"{'='*80}")

    for b in WORD_BINS:
        label = BIN_LABELS[b]
        n_bin = bin_counts[b]
        if n_bin == 0:
            continue

        total_correct = sum(results[b][d]['correct'] for d in [2, 4, 6, 8, 10])
        total_wrong   = sum(results[b][d]['wrong']   for d in [2, 4, 6, 8, 10])
        overall_pct   = total_correct / (total_correct + total_wrong) * 100 if (total_correct + total_wrong) > 0 else 0
        wa            = weighted_accuracy(results[b])

        print(f"\n  Žodžių: {label:>6}  |  deklaracijų: {n_bin:>6}  |  "
              f"bendras tiksl.: {overall_pct:.1f}%  |  svertinis tiksl.: {wa:.1f}%")
        print(f"    {'Gylis':>6}  {'Svoris':>6}  {'Teisingi':>10}  {'Neteisingi':>10}  {'Iš viso':>8}  {'%':>8}  {'Tikslumas':>8}")

        for d in [2, 4, 6, 8, 10]:
            c   = results[b][d]['correct']
            w   = results[b][d]['wrong']
            tot = c + w
            if tot == 0:
                continue
            pct_of_bin = tot / n_bin * 100
            acc        = c / tot * 100 if tot > 0 else 0.0
            print(f"    {f'{d} sk.':>6}  {WEIGHTS.get(d, '-'):>6}  {c:>10}  {w:>10}  {tot:>8}  ({pct_of_bin:>5.1f}%)  [{acc:>5.1f}%]")

    return results, bin_counts

results_rare,   bin_counts_rare   = evaluate_common_prefix_by_wordcount(mode='rare')
results_common, bin_counts_common = evaluate_common_prefix_by_wordcount(mode='common')

print(f"\n{'='*70}")
print(f"  RARE vs COMMON | svertinis tikslumas pagal žodžių skaičių")
print(f"  Svoriai: {dict(WEIGHTS)}")
print(f"{'='*70}")
print(
    f"{'Žodžių sk.':^15}"
    f"{'Deklaracijų sk.':^20}"
    f"{'RARE':^12}"
    f"{'COMMON':^12}"
    f"{'Geresnis':^12}"
)

for b in WORD_BINS:
    label = BIN_LABELS[b]
    n_bin = bin_counts_rare[b]
    if n_bin == 0:
        continue
    wa_r = weighted_accuracy(results_rare[b])
    wa_c = weighted_accuracy(results_common[b])
    winner = 'RARE' if wa_r > wa_c else ('COMMON' if wa_c > wa_r else '=')
    print(
        f"{label:^15}"
        f"{n_bin:^20}"
        f"{f'{wa_r:.1f}%':^12}"
        f"{f'{wa_c:.1f}%':^12}"
        f"{winner:^12}"
    )

In [ ]:
def format_label(val):
    return f'{val:.0f}%' if val % 1 == 0 else f'{val:.1f}%'.replace('.', ',')


def plot_wordcount_comparison(results_rare, results_common, bin_counts_rare):
    labels         = []
    wa_rare_vals   = []
    wa_common_vals = []

    for b in WORD_BINS:
        n_bin = bin_counts_rare[b]
        if n_bin == 0:
            continue
        labels.append(BIN_LABELS[b])
        wa_rare_vals.append(weighted_accuracy(results_rare[b]))
        wa_common_vals.append(weighted_accuracy(results_common[b]))

    x     = np.arange(len(labels))
    width = 0.35

    colors = {
        'RARE':   '#1c1c1c',
        'COMMON': '#8c8c8c',
    }

    fig, ax = plt.subplots(figsize=(12, 7))

    b1 = ax.bar(x - width/2, wa_rare_vals,   width, color=colors['RARE'],   zorder=3, label='$\it{Rare}$')
    b2 = ax.bar(x + width/2, wa_common_vals, width, color=colors['COMMON'], zorder=3, label='$\it{Common}$')

    for bar, val in zip(b1, wa_rare_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                format_label(val), ha='center', va='bottom', fontsize=10,
                color=colors['RARE'], fontweight='bold')

    for bar, val in zip(b2, wa_common_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                format_label(val), ha='center', va='bottom', fontsize=10,
                color='#505050', fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_xlabel('Žodžių skaičius deklaracijoje', fontsize=14, fontweight='bold')
    ax.set_ylabel('Svertinis tikslumas (%)', fontsize=14, fontweight='bold')
    ax.set_title('Svertinis tikslumas pagal aprašymų žodžių skaičiaus grupes', fontsize=16, fontweight='bold')
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(max(wa_rare_vals), max(wa_common_vals)) + 8)
    ax.legend(fontsize=14)

    plt.tight_layout()
    plt.show()

plot_wordcount_comparison(results_rare, results_common, bin_counts_rare)

Ši analizė nagrinėja, kaip modelio tikslumas priklauso nuo žodžio „kita“ (kamienas „kit“) pasikartojimo KN nomenklatūros aprašyme. Pirmiausia iš KN aprašymų išskiriamas šio žodžio kamienas ir kiekvienai deklaracijai apskaičiuojama, kiek kartų jis pasitaiko tiksliniame KN įraše. Deklaracijos tuomet sugrupuojamos pagal šio „kit“ kiekio intervalus (nuo 0 iki 4 ir daugiau), siekiant įvertinti, ar dažnesnis šio žodžio pasikartojimas daro įtaką klasifikavimo tikslumui. Kiekvienai grupei taikomas common-prefix modelis. Rezultatai vertinami tiek bendrai, tiek svertiniu tikslumu, ir palyginama tarp „rare“ ir „common“ modeliu.

In [ ]:
KIT_STEM = stemmer.stemWord("kita".translate(lt_letter_map))
print(f"'kita' kamienas: '{KIT_STEM}'")

def count_kit_in_kn(kn_df, true_code):
    kn_row = kn_df[kn_df['Goods code'].astype(str) == true_code]
    if kn_row.empty:
        return None
    text = str(kn_row['cleaned_description'].values[0])
    return sum(1 for w in text.split() if w == KIT_STEM)


def evaluate_common_prefix_by_kit(mode):
    df    = get_eval_df()
    total = len(df)

    kit_groups = {k: {d: {'correct': 0, 'wrong': 0} for d in [2, 4, 6, 8, 10]}
                  for k in [0, 1, 2, 3, 4, 5]}
    kit_totals = Counter()
    not_found  = 0

    for _, row in tqdm(df.iterrows(), total=total):
        year = row['Year']
        if year not in kn_dfs:
            continue

        kn_df   = kn_dfs[year]
        words   = str(row['cleaned_description']).split()
        true_10 = str(row['Kodas'])

        kit_count = count_kit_in_kn(kn_df, true_10)
        if kit_count is None:
            not_found += 1
            continue

        kit_count = min(kit_count, 5)
        kit_totals[kit_count] += 1

        selected = select_words(words, mode=mode)
        cands    = filter_candidates_unlimited(kn_df, selected)

        if len(cands) == 0:
            kit_groups[kit_count][2]['wrong'] += 1
            continue

        codes  = cands['Goods code'].astype(str).tolist()
        cp_len = common_prefix_len(codes)

        for lvl in [10, 8, 6, 4, 2]:
            if cp_len >= lvl:
                group = lvl
                break
        else:
            group = 2

        common_str = codes[0][:group] if group > 0 else ''
        correct    = true_10.startswith(common_str) if common_str else False

        if correct:
            kit_groups[kit_count][group]['correct'] += 1
        else:
            kit_groups[kit_count][group]['wrong']   += 1

    print(f"\n{'='*75}")
    print(f"  [{mode.upper()} | COMMON-PREFIX | pagal 'kit' kiekį KN apraše | n={total}]")
    print(f"  Svoriai: {dict(WEIGHTS)}")
    print(f"{'='*75}")

    for k in [0, 1, 2, 3, 4, 5]:
        n_kit = kit_totals[k]
        if n_kit == 0:
            continue

        buckets = kit_groups[k]
        label   = f"{k}+" if k == 5 else str(k)

        total_correct = sum(buckets[d]['correct'] for d in [2, 4, 6, 8, 10])
        total_wrong   = sum(buckets[d]['wrong']   for d in [2, 4, 6, 8, 10])
        overall_pct   = total_correct / (total_correct + total_wrong) * 100 if (total_correct + total_wrong) > 0 else 0

        wa = weighted_accuracy(buckets)

        print(f"\n  'kit' kiekis: {label}  |  deklaracijų: {n_kit:>7}  |  "
              f"bendras tiksl.: {overall_pct:.1f}%  |  svertinis tiksl.: {wa:.1f}%")
        print(f"    {'Gylis':>6}  {'Svoris':>6}  {'Teisingi':>10}  {'Neteisingi':>10}  {'Iš viso':>8}  {'%':>8}  {'Tikslumas':>8}")

        for d in [2, 4, 6, 8, 10]:
            c   = buckets[d]['correct']
            w   = buckets[d]['wrong']
            tot = c + w
            if tot == 0:
                continue
            pct_of_kit = tot / n_kit * 100
            acc        = c / tot * 100 if tot > 0 else 0.0
            print(f"    {f'{d} sk.':>6}  {WEIGHTS.get(d, '-'):>6}  {c:>10}  {w:>10}  {tot:>8}  ({pct_of_kit:>5.1f}%)  [{acc:>5.1f}%]")

    return kit_groups, kit_totals

kit_groups_rare,   kit_totals_rare   = evaluate_common_prefix_by_kit(mode='rare')
kit_groups_common, kit_totals_common = evaluate_common_prefix_by_kit(mode='common')

print(f"\n{'='*65}")
print(f"  RARE vs COMMON | svertinis tikslumas pagal 'kit' kiekį")
print(f"  Svoriai: {dict(WEIGHTS)}")
print(f"{'='*65}")
print(
    f"{'kit sk.':^10}"
    f"{'Deklaracijų sk.':^20}"
    f"{'RARE':^12}"
    f"{'COMMON':^12}"
    f"{'Geresnis':^12}"
)

for k in [0, 1, 2, 3, 4, 5]:
    n_kit = kit_totals_rare[k]
    if n_kit == 0:
        continue
    label = f"{k}+" if k == 5 else str(k)
    wa_r = weighted_accuracy(kit_groups_rare[k])
    wa_c = weighted_accuracy(kit_groups_common[k])
    winner = 'RARE' if wa_r > wa_c else ('COMMON' if wa_c > wa_r else '=')
    print(
        f"{label:^10}"
        f"{n_kit:^20}"
        f"{f'{wa_r:.1f}%':^12}"
        f"{f'{wa_c:.1f}%':^12}"
        f"{winner:^12}"
    )

In [ ]:
def plot_kit_comparison(kit_groups_rare, kit_groups_common, kit_totals_rare):
    kit_labels     = []
    wa_rare_vals   = []
    wa_common_vals = []

    for k in [0, 1, 2, 3, 4, 5]:
        n_kit = kit_totals_rare[k]
        if n_kit == 0:
            continue
        label = f"{k}+" if k == 5 else str(k)
        kit_labels.append(label)
        wa_rare_vals.append(weighted_accuracy(kit_groups_rare[k]))
        wa_common_vals.append(weighted_accuracy(kit_groups_common[k]))

    def format_label(val):
        return f'{val:.0f}%' if val % 1 == 0 else f'{val:.1f}%'.replace('.', ',')

    x     = np.arange(len(kit_labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(12, 7))

    colors = {
        'RARE':   '#1c1c1c',
        'COMMON': '#8c8c8c',
    }

    bars_rare   = ax.bar(x - width/2, wa_rare_vals,   width, color=colors['RARE'],   zorder=3, label='$\it{Rare}$')
    bars_common = ax.bar(x + width/2, wa_common_vals, width, color=colors['COMMON'], zorder=3, label='$\it{Common}$')

    for bar, val in zip(bars_rare, wa_rare_vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            format_label(val), ha='center', va='bottom', fontsize=10,
            color=colors['RARE'], fontweight='bold'
        )

    for bar, val in zip(bars_common, wa_common_vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            format_label(val), ha='center', va='bottom', fontsize=10,
            color='#505050', fontweight='bold'
        )

    ax.set_xticks(x)
    ax.set_xticklabels(kit_labels, fontsize=12)
    ax.set_xlabel('„kit" skaičius KN apraše', fontsize=14, fontweight='bold')
    ax.set_ylabel('Svertinis tikslumas (%)', fontsize=14, fontweight='bold')
    ax.set_title(
        'Rare ir Common modelių svertinis tikslumas pagal „kit" skaičių KN apraše',
        fontsize=16, fontweight='bold'
    )
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(max(wa_rare_vals), max(wa_common_vals)) + 8)
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

plot_kit_comparison(kit_groups_rare, kit_groups_common, kit_totals_rare)

Šiame etape analizuojama, kiek deklaracijų neturi nė vieno bendro žodžio su atitinkamu KN nomenklatūros aprašymu. Tam kiekvienos deklaracijos aprašymo žodžiai lyginami su tikrojo prekės kodo nomenklatūros aprašymo žodžiais, apskaičiuojant jų sankirtą. Jei bendrų žodžių nerandama, laikoma, kad containment reikšmė lygi nuliui. Analizė atliekama naudojant pradinius („žalio srauto“) duomenis su dubliais, todėl galima įvertinti, kokia dalis deklaracijų neturi tiesioginio tekstinio sutapimo su KN įrašais ir gali būti sudėtingesnės automatiniam klasifikavimui.

In [ ]:
raw_file = "combined_imports_cleaned_prepared_before_dedup.csv"

raw_df = pd.read_csv(raw_file)
raw_df['Year'] = pd.to_datetime(raw_df['Data']).dt.year

total      = len(raw_df)
zero_cont  = 0
no_kn_year = 0

for _, row in tqdm(raw_df.iterrows(), total=total):
    year = row['Year']
    if year not in kn_dfs:
        no_kn_year += 1
        continue

    decl_words = set(str(row['cleaned_description']).split())
    if not decl_words:
        zero_cont += 1
        continue

    kn_df     = kn_dfs[year]
    true_code = str(row['Kodas'])
    kn_row    = kn_df[kn_df['Goods code'].astype(str) == true_code]

    if kn_row.empty:
        zero_cont += 1
        continue

    kn_words = set(str(kn_row['cleaned_description'].values[0]).split())
    if len(decl_words & kn_words) == 0:
        zero_cont += 1

evaluated = total - no_kn_year
pct       = zero_cont / evaluated * 100 if evaluated > 0 else 0.0

print(f"\n{'='*55}")
print(f"  Containment = 0 analizė (žalio srauto duomenyse su dubliais)")
print(f"{'='*55}")
print(f"  Iš viso deklaracijų: {total:>8}")
print(f"  Containment = 0    : {zero_cont:>8}  ({pct:.2f}%)")
print(f"  Containment > 0    : {evaluated - zero_cont:>8}  ({100 - pct:.2f}%)")
print(f"{'='*55}")